# Predicting Online Shopper Purchase Intention Using Machine Learning

## Problem Statement

E-commerce businesses want to identify which website visitors are likely to make a purchase. Accurately predicting purchase intention allows companies to target high-intent customers, reduce bounce rates and improve conversion rates.

The objecive of this project is to build a machine learning classification model that predicts whether a user session will result in a purchase (Revenue = True) using behavioral metrics such as page views, duration, bounce rates and exit rates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv("online_shoppers_intention.csv")
df.head()

## Data Cleaning

The dataset was inspected for missing values, incorrect data types and duplicates. No significant data quality issues were found.

Categorical variables were encoded and numerical features were used directly. The dataset was ready for modeling with minimal preprocessing.

In [ ]:
df.columns

## Feature Engineering

New features were created to better capture user engagement and browsing behavior. These engineered features help improve model performance by combining related metrics.

Features created:
- TotalDuration: Sum of all page durations
- TotalPages: Total number of pages visited
- AvgDurationPerPage: Average time spent per page
- ProductFocus: Ratio of product-related pages
- BounceExitScore: Combined bounce and exit rates metric 

In [ ]:
## Total Engagement duration
df['TotalDuration'] = df['Administrative_Duration'] + df['Informational_Duration'] + df['ProductRelated_Duration']

## Total number of pages visited
df['TotalPages'] = df['Administrative'] + df['Informational'] + df['ProductRelated']

## Average duration per page
df['AvgDurationPerPage'] = df['TotalDuration'] / (df['TotalPages'] + 1)

## Product focus ratio (focus on product pages)
df['ProductFocus'] = df['ProductRelated'] / (df['TotalPages'] + 1)

## Bounce-exit combined score
df['BounceExitScore'] = df['BounceRates'] + df['ExitRates']

In [ ]:
# "numeric_only=True" ignores text columns and only calculates averages for numbers
df.groupby("Revenue").mean(numeric_only=True)

In [ ]:
print(df.columns)

In [ ]:
# Define X and y
features = ['Administrative', 'Informational', 'ProductRelated', 
            'Administrative_Duration', 'Informational_Duration', 
            'ProductRelated_Duration', 'BounceRates', 'ExitRates', 
            'PageValues', 'SpecialDay', 'TotalDuration', 'TotalPages', 
            'AvgDurationPerPage', 'ProductFocus', 'BounceExitScore']

X = df[features]
y = df['Revenue'].astype(int)
# convert True/False to 1/0

In [ ]:
# Converts categories into numbers using One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

## Train Test Split

The dataset was split into training and testing sets to evaluate model performance on unseen data. An 80/20 split was used, where 80% of the data was used for training and 20% for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Model Building

Two classification models were trained:

- Logistic Regression (Baseline model)
- Random Forest Classifier (ensemble model)

The goal was to compare performance and identify the better model for predicting purchase intention.

In [ ]:
# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
# Prediction
y_pred = model.predict(X_test)

## Threshold Tuning

Threshold tuning was applied to improve recall for the purchase class. Lowering the probability threshold allows the model to indentify more potential buyers at the cost of sightly reduced precision.

In [ ]:
# Predicting Probability of True Buyer
y_prob = model.predict_proba(X_test)[:,1]

# Modifying Threshold
y_pred_new = (y_prob >= 0.15).astype(int)

## Model Evaluation of Logistic Regression

Models were evaluated using multiple classification metrics:
- Confusion Matrix
- Precision
- Recall
- F1-score
- ROC Curve
- AUC Score

These metrics provide a comprehensive understanding of model performance, especially for the minority purchase class.

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_new))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_new))

In [ ]:
cm = confusion_matrix(y_test, y_pred_new)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

In [ ]:
# ROC Curve + AUC
# Receiver Operating Characteristics Curve + Area Under Curve

y_probs = model.predict_proba(X_test)[:,1]

fpr, tpr, threshold = roc_curve(y_test, y_probs)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.show()

print("AUC:")
print(roc_auc_score(y_test, y_probs))

In [ ]:
# Train Random Forest Model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Prediction
y_probs_rf = rf.predict_proba(X_test)[:,1]

# Threshold
y_pred_rf = (y_probs_rf >= 0.15).astype(int)

# Evaluation of Random Forest

In [ ]:
print("Classification Report")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion Matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp.plot()

In [ ]:
# ROC + AUC

fpr_rf, tpr_rf, threshold = roc_curve(y_test, y_probs_rf)

plt.plot(fpr_rf, tpr_rf)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.show()

print("AUC:")
print(roc_auc_score(y_test, y_probs_rf))

In [ ]:
df.columns

## Feature Importance

Feature importance was extracted from the Random Forest model to identify which variables had the greatest impact on predicting purchase intention.

In [ ]:
feature_names = X.columns

importances = rf.feature_importances_

feature_importance = pd.Series(importances, index=feature_names)

# Visualization
feature_importance.sort_values().plot(kind='barh')
plt.title("Feature Importance - Random Forest")
plt.show()

## Feature Insight Analysis

Boxplots were used to compare important features between buyers and non-buyers. This helps understand how user behavior differs and validates model findings.

In [ ]:
# Analyze Important Features

important_features = ["PageValues", "BounceExitScore", 
                     "ExitRates", "ProductRelated_Duration", 
                      "TotalDuration"]

for feature in important_features: 
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Revenue", y=feature, data=df)
    plt.title(f"{feature} vs Purchase")
    plt.show()

### Insight
- Customers who spend more time browsing product pages and have lower bounce and exit behavior are significantly more likely to complete a purchase. This suggests that engagement metrics are strong predictors of buying intent.


## Business Recommendations

Based on model insights, several business recommendations were developed to improve conversion rates and customer engagement.

### Recommendation 1 - Target Low Engagement Users

Users with:
 - High BounceExitScore
 - High ExitRates
 - Low duration

Are unlikely to buy.

### Action:

- Show pop-up discount
- Offer free shipping
- Show limited-time offer

Goal: keep them from leaving.

### Recommendation 2 - Focus on Highly Engaged Users

Users with:
- High ProductRelated_Duration
- High TotalDuration
- High PageValues

Are close to buying.

### Action:

- Show "Customers also bought"
- Show product reviews
- Offer bundle deals

Goal: push them to complete purchase.

### Recommendation 3 - Real-Time Purchase Prediction

This model can run live on website.

When probability > threshold:

- Trigger personalized offer
- Highlight checkout button
- Show urgency ("Only 3 left")

This increases conversion rate.

### Recommendation 4 - Improve Product Pages

Since ProductRelated_Duration is important:

Businesses should:

- Add better images
- Add videos
- Add reviews
- Add comparison tables

Goal: keep users engaged longer.

## Conclusion

This project developed a machine learning model to predict online shopper purchase intention using behavioral session data.

Feature importance analysis showed that engagement-related variables such as PageValues, BounceExitScore, ExitRates, ProductRelated_Duration and TotalDuration were the strongest predictors of purchase behavior. Customers who spent more time browsing product_related pages and exhibited lower bounce and exit rates were significatly more likely to complete a purchase.

The Random Forest model outperformed Logistic Regression and demonstrated strong predictive capability. Threshold tuning improved recall for the minority purchase class, making the model more useful for identifying potential buyers.

Business recommendations include targeting low-engagement users with retention offers, providing personalized recommendations to highly engaged users and using the model in real-time to trigger marketing actions. These strategies can help increase conversion rates and improve overall customer engagement.

Overall, the project demonstrates how behavioral analytics and machine learning can be combined to generate actionable insights and support data-driven decision making in e-commerce environments.